In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [3]:
CSV_PATH = 'dataset/raw_car_dataset.csv'
df = pd.read_csv(CSV_PATH)

In [4]:
print(df.head(10))

    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022
5  $1,500     211171.0    4.0          0       ??  2003
6  1500.0     222235.0    4.0          2    Rural  2004
7  $1,500     105068.0    5.0          1     City  2002
8  $2,291      90015.0    4.0          0    Rural  2007
9  1500.0     125976.0    2.0          0     City  2002


In [5]:
print(df.shape)
 

(145, 6)


In [6]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    object 
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    object 
 5   Year         145 non-null    int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 6.9+ KB
None


In [7]:
print(df.isnull().sum())

Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64


In [8]:

print(df["Location"].value_counts(dropna=False))

Location
City      59
Suburb    45
Rural     21
Subrb      8
??         7
NaN        5
Name: count, dtype: int64


In [9]:
df["Price"] = df["Price"].replace({r"[\$,]": ""} , regex=True).astype(float)

In [13]:
print(df["Price"].dtype)
print(df["Price"].skew())

float64
7.871113660850301


In [14]:
df["Location"] = df["Location"].replace({
    "Subrb": "Suburb",
    "CITY": "City",
    "city": "City",
    "??": pd.NA
})

print(df["Location"].value_counts(dropna=False))

Location
City      59
Suburb    53
Rural     21
<NA>       7
NaN        5
Name: count, dtype: int64


In [15]:
df["Odometer_km"] = df["Odometer_km"].fillna(df["Odometer_km"].median())
df["Doors"] = df["Doors"].fillna(df["Doors"].mode()[0])
df["Location"] = df["Location"].fillna(df["Location"].mode()[0])

In [16]:
print(df.isnull().sum())

Price          0
Odometer_km    0
Doors          0
Accidents      0
Location       0
Year           0
dtype: int64


In [18]:
before = df.shape

df = df.drop_duplicates()

after = df.shape

print("Before:", before)
print("After :", after)

Before: (140, 6)
After : (140, 6)


In [13]:
def iqr_fun(series, k+1.5)

In [20]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper
low_price, high_price = iqr_fun(df["Price"])
low_odometer, high_odometer = iqr_fun(df["Odometer_km"])

In [21]:
print("IQR of Price")
print("Lower:", low_price, " Upper:", high_price)

print()

print("IQR of Odometer")
print("Lower:", low_odometer, " Upper:", high_odometer)

IQR of Price
Lower: -2984.625  Upper: 8974.375

IQR of Odometer
Lower: -6642.25  Upper: 271987.75


In [22]:
df["Price"] = df["Price"].clip(low_price, high_price)

df["Odometer_km"] = df["Odometer_km"].clip(low_odometer, high_odometer)

print(df[["Price", "Odometer_km"]].describe())

             Price    Odometer_km
count   140.000000     140.000000
mean   3175.456250  130945.403571
std    2601.848629   53815.006935
min    1500.000000    5000.000000
25%    1500.000000   97844.000000
50%    1500.000000  128548.000000
75%    4489.750000  167501.500000
max    8974.375000  271987.750000


In [23]:
df = pd.get_dummies(df, columns=["Location"], dtype=int)

print(df.columns)

Index(['Price', 'Odometer_km', 'Doors', 'Accidents', 'Year', 'Location_City',
       'Location_Rural', 'Location_Suburb'],
      dtype='object')


In [24]:
current_year = 2026

df["CarAge"] = current_year - df["Year"]

df["Km_per_year"] = df["Odometer_km"] / df["CarAge"].replace(0, 1)

if "Location_City" in df.columns and "Location_Suburb" in df.columns:
    df["Is_Urban"] = (
        (df["Location_City"] == 1) |
        (df["Location_Suburb"] == 1)
    ).astype(int)

df["LogPrice"] = np.log(df["Price"] + 1)

print(df.head())

    Price  Odometer_km  Doors  Accidents  Year  Location_City  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998              1               0   
1  4171.0     128548.0    4.0          0  2016              0               1   
2  5331.0     107302.0    4.0          0  2014              0               0   
3  1500.0     141838.0    4.0          1  1999              0               0   
4  1500.0     128548.0    3.0          0  2022              1               0   

   Location_Suburb  CarAge   Km_per_year  Is_Urban  LogPrice  
0                0      28   4922.500000         1  7.313887  
1                0      10  12854.800000         0  8.336151  
2                1      12   8941.833333         1  8.581482  
3                1      27   5253.259259         1  7.313887  
4                0       4  32137.000000         1  7.313887  


In [25]:
scaler = StandardScaler()

continuous_features = [
    "Odometer_km",
    "Doors",
    "Accidents",
    "Year",
    "CarAge",
    "Km_per_year"
]

df[continuous_features] = scaler.fit_transform(df[continuous_features])

print(df[continuous_features].head())

   Odometer_km     Doors  Accidents      Year    CarAge  Km_per_year
0     0.128390  0.254091   0.316968 -1.686714  1.686714    -0.615631
1    -0.044709  0.254091  -0.820867  0.794617 -0.794617     0.070446
2    -0.440923  0.254091  -0.820867  0.518913 -0.518913    -0.267993
3     0.203135  0.254091   0.316968 -1.548862  1.548862    -0.587024
4    -0.044709 -0.931668  -0.820867  1.621727 -1.621727     1.738196


In [26]:
print(df.info())

print(df.isnull().sum())

print(df.describe())

df.to_csv("clean_car_dataset.csv", index=False)

print("Dataset saved successfully.")

<class 'pandas.core.frame.DataFrame'>
Index: 140 entries, 0 to 139
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Price            140 non-null    float64
 1   Odometer_km      140 non-null    float64
 2   Doors            140 non-null    float64
 3   Accidents        140 non-null    float64
 4   Year             140 non-null    float64
 5   Location_City    140 non-null    int64  
 6   Location_Rural   140 non-null    int64  
 7   Location_Suburb  140 non-null    int64  
 8   CarAge           140 non-null    float64
 9   Km_per_year      140 non-null    float64
 10  Is_Urban         140 non-null    int64  
 11  LogPrice         140 non-null    float64
dtypes: float64(8), int64(4)
memory usage: 14.2 KB
None
Price              0
Odometer_km        0
Doors              0
Accidents          0
Year               0
Location_City      0
Location_Rural     0
Location_Suburb    0
CarAge             0
Km_per_year

In [27]:
df.to_csv("clean_car_dataset.csv", index=False)